## Setup

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "1.0-POC"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


In [7]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline
from LLMDialogue.models.gpt2 import GPT2

In [8]:
gpt2 = GPT2(seed=GLOBAL_SEED)
# Simply the 
gpt2.load(
    #model_checkpoint="/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/pipelines/7d3862ab4a7d70098663df37d07f89108065f8a31cac420f0631e6d4ccfdd356/checkpoint-335", 
)

In [9]:
gpt2.model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50258, bias=False)
)

In [10]:
# Create the pipeline config 
config = CPPipelineConfig(
    model = gpt2, 
    utterances=['this is is a text'],
    N = -1,
    context_buffer_size=512,   
)

In [11]:
# Now, init the pipeline 
pipe = CPPipeline(config)


In [12]:
# Run the pipeline 
pipe.run()

Processing turns: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


In [13]:
pipe.load_from_disk()

{'configs': {'date_created': '22_05_2024__15_46_50',
  'name': 'ConditionalProbabilityPipeline',
  'variant_name': 'default',
  'version': '0.0.0',
  'data_dir': None,
  'data_files': None,
  'cache_dir': '/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/pipelines/c3d1bf21182093b64bf40cbc4abd3c3c1ac31a3ae73b6bc4a1f70fdb70f4dc63',
  'clear_cache': False,
  'config_id': 'c3d1bf21182093b64bf40cbc4abd3c3c1ac31a3ae73b6bc4a1f70fdb70f4dc63',
  'archive_dir': '/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/pipelines/c3d1bf21182093b64bf40cbc4abd3c3c1ac31a3ae73b6bc4a1f70fdb70f4dc63/archive',
  'compress_archive': True,
  'seed': 42,
  'verbose': False,
  'device': None,
  'model': "<class 'LLMDialogue.models.gpt2.gpt2.GPT2'>",
  'utterances': ['this is is a text'],
  'N': -1,
  'context_buffer_size': 512},
 'data': [{'turn_no': 0,
   'word_no': 1,
   'context': 'this',
   'word': 'is',
   'last_word_prob': 1.21

In [14]:


from LLMDialogue.models import GPT2, TurnGPT
from LLMDialogue.pipelines import LLMFinetunePipeline, LLMFinetunePipelineConfig
from LLMDialogue.dm.ICCdm import ICCHFLMDM, ICCSpokenLanguageModelingDM
from LLMDialogue.utils.MLTrackers import WANDBTracker

In [15]:
# gpt2 = GPT2(seed=GLOBAL_SEED)
# # Simply the 
# gpt2.load()

model = TurnGPT(seed=GLOBAL_SEED)
model.load()

/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers
Initialized special wte for <ts> to average of ['!', '?', '.']
This will make the <ts> semantically closer to ['!', '?', '.']


In [16]:
# dm = ICCHFLMDM(
#     dset_variant="labels", 
#     tokenizer=gpt2.tokenizer, 
#     save_dir=OUTPUT_DIR
# )

# Load the appropriate dm and finetune
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=model.tokenizer,
    save_dir=OUTPUT_DIR
)

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/743 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/802 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/591 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/859 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

In [17]:
run_name = "test"
config = LLMFinetunePipelineConfig(
    model=model, 
    dm=dm,
    finetune_kwargs={
        # "num_train_epochs" : 1,
        # "run_name" : run_name,
        "max_epochs" : 1
    },
    report_to="wandb",

)

In [18]:

pipe = LLMFinetunePipeline(config)

In [19]:
wandb_tracker = WANDBTracker()

In [20]:
wandb_tracker.init(
    project="gpt_monologue_to_dialogue", 
    entity="hilab", 
    mode="online", 
    run_name="test", 
    log_model="end",
    logging_dir=config.cache_dir
)
pipe.run()
wandb_tracker.finish_run()

2024-05-22 15:46:54,182 | ERROR | jupyter | Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: umair-aarij (hilab). Use `wandb login --relogin` to force relogin


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

...

/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/src/LLMDialogue/dm/spokenLanguageDM.py:136: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ret[k] = torch.tensor(v)
/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/src/LLMDialogue/dm/spokenLanguageDM.py:136: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ret[k] = torch.tensor(v)
/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/src/LLMDialogue/dm/spokenLanguageDM.py:136: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ret[k] = torch.tensor(v)


Validation: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,▁
train/loss,▅█▃▃▁▂▂▂▁▂▂▂▂▂▁▂▂▁▁▂▂▂▁▂▂▁▂▂▁▂▂▁▁▁▂▁▁▂▂▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch,0
eval/loss,1.45725
train/loss,0.00043
trainer/global_step,260


In [23]:
artifact_path = wandb_tracker.download_artifact(
    project="gpt_monologue_to_dialogue", 
    entity="hilab", 
    # artifact_name=f"model-{run_name}",
    artifact_name="model-ymwwgwhm",
    alias="v0",
    download_path=config.cache_dir
)
artifact_path

wandb: Downloading large artifact model-ymwwgwhm:v0, 1425.73MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:0.7


'/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/pipelines/025f8aed6ba0094e3de39aad43dd38b60327b5ceac46d6f687d5b665bf26fc70'

In [26]:

model.load("/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/pipelines/025f8aed6ba0094e3de39aad43dd38b60327b5ceac46d6f687d5b665bf26fc70/model.ckpt")



/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers
Initialized special wte for <ts> to average of ['!', '?', '.']
This will make the <ts> semantically closer to ['!', '?', '.']


In [29]:
model

Base model: TurnGPTLMHeadModel(
  (_transformer): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50260, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D()
            (c_proj): Conv1D()
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=768, out_features=50260, bias=False)
  )
)
Base tok

In [ ]:
# Really, the thing we need to predict on 




In [ ]:
wandb_tracker = WANDBTracker(
    api_key="e25e51de3749b5556e2d8b3d77f2bd8a0760d1ae", 
    project="gpt_monologue_to_dialogue", 
    entity="hilab", 
    mode="online", 
    run_name="test", 
    log_model="end",
    logging_dir=config.cache_dir
)

In [ ]:
wandb_tracker.start()
pipe.run()
wandb_tracker.finish()

In [ ]:

wandb_tracker = WANDBTracker(
    api_key="e25e51de3749b5556e2d8b3d77f2bd8a0760d1ae", 
    project="gpt_monologue_to_dialogue", 
    entity="hilab", 
    mode="online", 
    run_name="test", 
    log_model="end",
    logging_dir=config.cache_dir
)
wandb_tracker.start()
artifact = wandb_tracker.run.use_artifact("model-agb7i2zq:v1")

In [ ]:
artifact.download()

In [ ]:
import wandb 

In [ ]:
wandb.finish()

## Datasets

### In Conversation Corpus (ICC) 

#### Variant - No Labels

This variant of the ICC does not include any conversation start or end or 
speaker labels. 

In [ ]:
from LLMDialogue.datasets import load
from LLMDialogue.datasets.ICC import objectify as ICCObjectify, ICCDatasetSplits

In [ ]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
print(dset)

In [ ]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names


In [ ]:
dset_splits.data_df_map['train']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


#### Variant - Special Labels

In [ ]:
dset = load(
    dset_name="ICC", 
    config_name="labels"
)
print(dset)

In [ ]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['train']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


### Speaker Identity

#### Variant - No Labels

In [ ]:
from LLMDialogue.datasets.speakerIdentity import SpeakerIdentityDataSplits, objectify as SIObjectify

In [ ]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [ ]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['test']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


#### Labels

In [ ]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [ ]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

In [ ]:
dset_splits.data_df_map['test']

In [ ]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)

### Data Modules

These are required to be able to train the model 

In [ ]:
from LLMDialogue.dm.ICCdm import ICCHFLMDM
from LLMDialogue.dm.HFdm import HFLanguageModelingDM
from transformers import AutoTokenizer
from LLMDialogue.dm.ICCdm import ICCSpokenLanguageModelingDM
from LLMDialogue.tokenizers import SpokenDialogueTokenizer

#### ICC Language Modeling Data Module (Compatible with HuggingFace)

In [ ]:
# Load the tokenizer here
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=tokenizer, 
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

#### ICC Spoken Language Modeling Data Module (Compatible with PL)

In [ ]:
tokenizer = SpokenDialogueTokenizer()

In [ ]:
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

### Fine-tuning

#### GPT-2

The first model we want to funetune is GPT2, which is an instance of a 
language model from hugging face. 

In [ ]:
from LLMDialogue.models import GPT2

In [ ]:
# Instantiate and load the model 
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()
gpt2.to(GLOBAL_DEVICE)
print(gpt2)

In [ ]:
# Now, finetune the model using the appropriate data module 
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=gpt2.tokenizer, 
    save_dir=OUTPUT_DIR
)


In [ ]:
# Now, finetune the model. 
# gpt2.finetune(
#     dm=dm, 
#     save_dir=f"{OUTPUT_DIR}/models/gpt2",
#     num_train_epochs=1
# )

### TurnGPT

In [ ]:
from LLMDialogue.models import TurnGPT 


In [ ]:
turngpt = TurnGPT(seed=GLOBAL_SEED)
turngpt.load(model_head="LMHead")
turngpt.to(device=GLOBAL_DEVICE)
print(turngpt)

In [ ]:
# Load the appropriate dm and finetune
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)

In [ ]:
# turngpt.finetune(
#     dm=dm,
#     save_dir=f"{OUTPUT_DIR}/models/turngpt",
#     max_epochs=1
# )

### Pipelines

In [ ]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline

In [ ]:
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()

In [ ]:
# Create the pipeline config 
config = CPPipelineConfig(
    model = gpt2, 
    utterances=['this is is a text'],
    N = -1,
    context_buffer_size=512,   
)

In [ ]:
# Now, init the pipeline 
pipe = CPPipeline(config)


